In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import types

In [2]:
MYSQL_HOST = 'bsan6060-2023.c9amrakftmiu.us-west-2.rds.amazonaws.com'
MYSQL_USER = 'mchin'
MYSQL_PASSWORD = 'pw4AWSassign2'
MYSQL_DB = 'mchin_assign2'

uri = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}/{MYSQL_DB}'
engine = create_engine(uri, pool_timeout=1200)

## pageviews

In [3]:
pageviews = pd.read_csv('website_pageviews.csv')

In [7]:
pageviews[pageviews.duplicated()]

,website_pageview_id,created_at,website_session_id,pageview_url


In [9]:
pageviews.isnull().sum()

website_pageview_id    0
created_at             0
website_session_id     0
pageview_url           0
dtype: int64

In [11]:
pageviews['pageview_url'].unique().tolist()

['/home',
 '/products',
 '/original-gift-basket',
 '/cart',
 '/shipping',
 '/billing',
 '/thank-you',
 '/lp-1',
 '/billing-2',
 '/valentines-gift-basket',
 '/lp-2',
 '/lp-3',
 '/birthday-gift-basket',
 '/lp-4',
 '/lp-5',
 '/holiday-gift-basket']

In [13]:
pageviews.to_sql(name='pageviews', con=engine, if_exists='replace', chunksize=100000, index=False,
        dtype={
            'website_pageview_id': types.Integer,
            'created_at': types.TIMESTAMP,
            'website_session_id': types.Integer,
            'pageview_url': types.VARCHAR(50)
        }
    )

1188124

## sessions

In [156]:
SOURCE_HOST = 'db.isba.co'
SOURCE_USER = 'analyst'
SOURCE_PASSWORD = 'go_lions'
SOURCE_DB = 'basket_craft'

source_uri = f'mysql+pymysql://{SOURCE_USER}:{SOURCE_PASSWORD}@{SOURCE_HOST}/{SOURCE_DB}'
source_engine = create_engine(source_uri, pool_timeout=1200)

In [158]:
query = "SELECT * FROM website_sessions"
sessions = pd.read_sql(query, source_engine)

In [159]:
sessions[sessions.duplicated()]

,website_session_id,created_at,user_id,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer


In [160]:
sessions.isnull().sum()

website_session_id        0
created_at                0
user_id                   0
is_repeat_session         0
utm_source            83328
utm_campaign          83328
utm_content           83328
device_type               0
http_referer          39917
dtype: int64

In [161]:
sessions['utm_source'].unique()

array(['google', None, 'bing', 'facebook'], dtype=object)

In [162]:
sessions['utm_source'] = sessions['utm_source'].fillna('other')

In [164]:
sessions['utm_campaign'].unique()

array(['nonbrand', None, 'brand', 'pilot', 'desktop_targeted'],
      dtype=object)

In [165]:
sessions['utm_campaign'] = sessions['utm_campaign'].fillna('other')

In [166]:
sessions['utm_content'].unique()

array(['g_ad_1', None, 'g_ad_2', 'b_ad_2', 'b_ad_1', 'social_ad_1',
       'social_ad_2'], dtype=object)

In [167]:
sessions['utm_content'] = sessions['utm_content'].fillna('other')

In [168]:
sessions['http_referer'].unique()

array(['https://google.com', None, 'https://bing.com',
       'https://facebook.com'], dtype=object)

In [169]:
sessions['http_referer'] = sessions['http_referer'].fillna('other')

In [170]:
sessions.isnull().sum()

website_session_id    0
created_at            0
user_id               0
is_repeat_session     0
utm_source            0
utm_campaign          0
utm_content           0
device_type           0
http_referer          0
dtype: int64

In [171]:
sessions.to_sql(name='sessions', con=engine, if_exists='replace', chunksize=100000, index=False,
        dtype={
            'website_session_id': types.INT,
            'created_at': types.TIMESTAMP,
            'user_id': types.INT,
            'is_repeat_session': types.SMALLINT,
            'utm_source': types.VARCHAR(12),
            'utm_campaign': types.VARCHAR(20),
            'utm_content': types.VARCHAR(15),
            'device_type': types.VARCHAR(15),
            'http_referer': types.VARCHAR(30)
        }
    )

472871

## fact_order_item

In [184]:
# Write your query
query = """
SELECT 
    o.order_id,
    oi.order_item_id,
    p.product_id,
    DATE(o.created_at) AS order_date,
    u.user_id,
    SUM(oi.price_usd * o.items_purchased) AS revenue_usd,
    SUM(oi.cogs_usd * o.items_purchased) AS cogs_usd,
    SUM((oi.price_usd * o.items_purchased) - (oi.cogs_usd * o.items_purchased)) AS profit_usd
FROM 
    order_items AS oi
JOIN orders AS o
    ON oi.order_id = o.order_id
JOIN products AS p
    ON oi.product_id = p.product_id
JOIN users AS u
    ON o.user_id = u.user_id
JOIN dim_date AS d
    ON DATE(o.created_at) = d.date
GROUP BY 
    o.order_id,
    oi.order_item_id,
    p.product_id,
    d.date,
    u.user_id,
    DATE(o.created_at);
"""

# Execute and fetch data
fact_order_item = pd.read_sql(query, con=engine)

# Preview data
fact_order_item.head()


,order_id,order_item_id,product_id,order_date,user_id,revenue_usd,cogs_usd,profit_usd
0,1,1,1,2021-03-19,20,49.99,19.49,30.5
1,2,2,1,2021-03-20,104,49.99,19.49,30.5
2,3,3,1,2021-03-20,147,49.99,19.49,30.5
3,4,4,1,2021-03-20,160,49.99,19.49,30.5
4,5,5,1,2021-03-20,177,49.99,19.49,30.5


In [185]:
fact_order_item.to_sql(name='fact_order_item', con=engine, if_exists='replace', chunksize=100000, index=False,
        dtype={
            'order_id': types.INT,
            'order_item_id': types.INT,
            'product_id': types.INT,
            'order_date': types.DATE,
            'user_id': types.INT,
            'revenue_usd': types.DECIMAL(10,2),
            'cogs_usd': types.DECIMAL(10,2),
            'profit_usd': types.DECIMAL(10,2)
        }
    )

40025

## fact_order_item_refund

In [186]:
# Write your query
query = """
SELECT 
    oi.order_item_id,
    oir.order_item_refund_id,
    p.product_id,
    DATE(oir.created_at) AS refund_date,
    u.user_id,
    SUM(oir.refund_amount_usd) AS refund_usd,
    SUM(oi.cogs_usd * o.items_purchased) AS cogs_usd,
    SUM(oir.refund_amount_usd - (oi.cogs_usd * o.items_purchased)) AS profit_loss_usd
FROM 
    order_item_refunds AS oir
JOIN order_items AS oi
    ON oir.order_item_id = oi.order_item_id
JOIN orders AS o
    ON oi.order_id = o.order_id
JOIN products AS p
    ON oi.product_id = p.product_id
JOIN users AS u
    ON o.user_id = u.user_id
JOIN dim_date AS d
    ON DATE(oir.created_at) = d.date
GROUP BY 
    oi.order_item_id,
    oir.order_item_refund_id,
    p.product_id,
    refund_date,
    u.user_id;

"""

# Execute and fetch data
fact_order_item_refund = pd.read_sql(query, con=engine)

# Preview data
fact_order_item_refund.head()


,order_item_id,order_item_refund_id,product_id,refund_date,user_id,refund_usd,cogs_usd,profit_loss_usd
0,57,1,1,2021-04-06,1794,49.99,19.49,30.5
1,74,2,1,2021-04-13,2309,49.99,19.49,30.5
2,71,3,1,2021-04-15,2247,49.99,19.49,30.5
3,118,4,1,2021-04-18,4065,49.99,19.49,30.5
4,116,5,1,2021-04-23,3895,49.99,19.49,30.5


In [187]:
fact_order_item_refund.to_sql(name='fact_order_item_refund', con=engine, if_exists='replace', chunksize=100000, index=False,
        dtype={
            'order_item_id': types.INT,
            'order_item_refund_id': types.INT,
            'product_id': types.INT,
            'user_id': types.INT,
            'refund_date': types.DATE,
            'refund_usd': types.DECIMAL(10,2),
            'cogs_usd': types.DECIMAL(10,2),
            'profit_loss_usd': types.DECIMAL(10,2)
        }
    )

1707

## dim_landing_page

In [81]:
# Write your query
query = """
SELECT DISTINCT pageview_url AS landing_page_url
FROM pageviews;
"""

# Execute and fetch data
dim_landing_page = pd.read_sql(query, con=engine)

# Preview data
dim_landing_page

,landing_page_url
0,/home
1,/products
2,/original-gift-basket
3,/cart
4,/shipping
5,/billing
6,/thank-you
7,/lp-1
8,/billing-2
9,/valentines-gift-basket


In [83]:
dim_landing_page.to_sql(name='dim_landing_page', con=engine, if_exists='replace', chunksize=100000, index=False,
        dtype={
            'landing_page_url': types.VARCHAR(50)
        }
    )

16

## fact_website_sessions

In [205]:
# Step 1: Get the first pageview for each session
first_pageviews_query = """
SELECT
    p.website_session_id,
    p.pageview_url
FROM
    pageviews p
JOIN (
    SELECT website_session_id, MIN(website_pageview_id) AS first_pageview_id
    FROM pageviews
    GROUP BY website_session_id
) AS first_pageview ON p.website_pageview_id = first_pageview.first_pageview_id
"""

# Execute query to get first pageview per session
first_pageviews_df = pd.read_sql(first_pageviews_query, con=engine)

# Preview the first few rows
first_pageviews_df.head()


,website_session_id,pageview_url
0,1,/home
1,2,/home
2,3,/home
3,4,/home
4,5,/home


In [206]:
first_pageviews_df.to_sql(name='first_pageview', con=engine, if_exists='replace', chunksize=100000, index=False,
        dtype={
            'website_session_id': types.INT,
            'pageview_url': types.VARCHAR(50)
        }
    )

472871

In [225]:
main_query = """
SELECT
    s.website_session_id,
    d.`date` AS session_date,
    f.pageview_url AS landing_page_url,
    dc.campaign_id,
    s.user_id,.
    s.device_type,
    COUNT(DISTINCT p.website_pageview_id) AS pageviews_count,
    COUNT(DISTINCT o.order_id) AS orders_count,
    SUM(CASE WHEN s.is_repeat_session = 1 THEN 1 ELSE 0 END) AS bounce_count
FROM
    sessions s
JOIN
    pageviews p ON s.website_session_id = p.website_session_id
JOIN
    first_pageview f ON s.website_session_id = f.website_session_id
LEFT JOIN
    orders o ON s.website_session_id = o.website_session_id
JOIN
    dim_date d ON DATE(s.created_at) = d.date
LEFT JOIN
    dim_campaign dc ON 
        s.utm_campaign = dc.utm_campaign AND
        s.utm_source   = dc.utm_source AND
        s.utm_content  = dc.utm_content
LEFT JOIN
    dim_user du ON s.user_id = du.user_id
GROUP BY
    s.website_session_id, d.`date`, f.pageview_url, dc.campaign_id, s.user_id, s.device_type
"""


fact_website_sessions = pd.read_sql(main_query, con=engine)

fact_website_sessions


,website_session_id,session_date,landing_page_url,campaign_id,user_id,product_id,device_type,pageviews_count,orders_count,bounce_count
0,1,2021-03-19,/home,1,1,NaN,mobile,1,0,0.0
1,2,2021-03-19,/home,1,2,NaN,desktop,1,0,0.0
2,3,2021-03-19,/home,1,3,NaN,desktop,1,0,0.0
3,4,2021-03-19,/home,1,4,NaN,desktop,1,0,0.0
4,5,2021-03-19,/home,1,5,NaN,mobile,1,0,0.0
...,...,...,...,...,...,...,...,...,...,...
472866,472867,2024-03-19,/home,3,394314,NaN,desktop,2,0,0.0
472867,472868,2024-03-19,/lp-3,5,394315,NaN,mobile,3,0,0.0
472868,472869,2024-03-19,/lp-3,1,394316,NaN,mobile,1,0,0.0
472869,472870,2024-03-19,/lp-5,1,394317,NaN,desktop,3,0,0.0


In [191]:
from sqlalchemy import text

with engine.connect() as connection:
    connection.execute(text("SET foreign_key_checks = 0;"))

In [192]:
fact_website_sessions.to_sql(name='fact_website_sessions', con=engine, if_exists='append', chunksize=100000, index=False,
        dtype={
            'website_session_id': types.INT,
            'session_date': types.DATE,
            'landing_page_url': types.VARCHAR(50),
            'campaign_id': types.INT,
            'user_id': types.INT,
            'device_type': types.VARCHAR(15),
            'pageviews_count': types.INT,
            'orders_count': types.INT,
            'bounce_count': types.INT
        }
    )

472871

In [193]:
with engine.connect() as connection:
    connection.execute(text("SET foreign_key_checks = 1;"))